# PROYECTO FINAL

# Obtenemos Datos con API

In [3]:
import os

# El nombre de usuario es el que aparece en tu URL de Kaggle (ej: 'juanperez')
os.environ['KAGGLE_USERNAME'] = "danieldelgado23"

# Aquí pega el código largo (Token) que te generó la página de tu imagen
os.environ['KAGGLE_KEY'] = "KGAT_d569c08c514f70167820700b42051e32"

In [4]:
import kaggle

# Definimos el dataset y la ruta donde lo quieres guardar
dataset = 'shekpaul/global-superstore'
ruta = '../data/raw/'

# Descarga y descomprime
kaggle.api.dataset_download_files(dataset, path=ruta, unzip=True)

print("¡Listo! El archivo ya debe estar en tu carpeta ../data/raw/")

Dataset URL: https://www.kaggle.com/datasets/shekpaul/global-superstore
¡Listo! El archivo ya debe estar en tu carpeta ../data/raw/


In [5]:
import pandas as pd

# 1. Ruta corregida (Saliendo de 'src')
file_path = '../data/raw/Global Superstore.xls'

# 2. Intentamos leer como Excel real (necesitas instalar: pip install xlrd)
try:
    df = pd.read_excel(file_path)
    print("¡Cargado como Excel!")
except:
    # 3. Si falla, es que es un HTML disfrazado
    try:
        tablas = pd.read_html(file_path)
        df = tablas[0]
        print("¡Cargado como Tabla HTML!")
    except Exception as e:
        print(f"Sigue fallando. Error: {e}")

# Si logramos cargar algo, limpiamos nombres de columnas
if 'df' in locals():
    df.columns = [col.strip().replace(' ', '_').lower() for col in df.columns]
    display(df.head())

¡Cargado como Excel!


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,city,state,...,product_id,category,sub-category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority
0,32298,CA-2012-124891,2012-07-31,2012-07-31,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,...,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,2309.650,7,0.0,762.1845,933.57,Critical
1,26341,IN-2013-77878,2013-02-05,2013-02-07,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,...,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,0.1,-288.7650,923.63,Critical
2,25330,IN-2013-71249,2013-10-17,2013-10-18,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,...,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,0.1,919.9710,915.49,Medium
3,13524,ES-2013-1579342,2013-01-28,2013-01-30,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,...,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless",2892.510,5,0.1,-96.5400,910.16,Medium
4,47221,SG-2013-4320,2013-11-05,2013-11-06,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,...,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.960,8,0.0,311.5200,903.04,Critical


# AHORA SI!

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   row_id          51290 non-null  int64         
 1   order_id        51290 non-null  str           
 2   order_date      51290 non-null  datetime64[us]
 3   ship_date       51290 non-null  datetime64[us]
 4   ship_mode       51290 non-null  str           
 5   customer_id     51290 non-null  str           
 6   customer_name   51290 non-null  str           
 7   segment         51290 non-null  str           
 8   city            51290 non-null  str           
 9   state           51290 non-null  str           
 10  country         51290 non-null  str           
 11  postal_code     9994 non-null   float64       
 12  market          51290 non-null  str           
 13  region          51290 non-null  str           
 14  product_id      51290 non-null  str           
 15  category     

- Tenemos un data set de 24 columnas y 51.290 filas

In [7]:
# Analisis Descriptivo

df.describe()

,row_id,order_date,ship_date,postal_code,sales,quantity,discount,profit,shipping_cost
count,51290.00000,51290,51290,9994.000000,51290.000000,51290.000000,51290.000000,51290.000000,51290.000000
mean,25645.50000,2013-05-11 21:26:49.155780,2013-05-15 20:42:42.745174,55190.379428,246.490581,3.476545,0.142908,28.610982,26.375818
min,1.00000,2011-01-01 00:00:00,2011-01-03 00:00:00,1040.000000,0.444000,1.000000,0.000000,-6599.978000,0.002000
25%,12823.25000,2012-06-19 00:00:00,2012-06-23 00:00:00,23223.000000,30.758625,2.000000,0.000000,0.000000,2.610000
50%,25645.50000,2013-07-08 00:00:00,2013-07-12 00:00:00,56430.500000,85.053000,3.000000,0.000000,9.240000,7.790000
75%,38467.75000,2014-05-22 00:00:00,2014-05-26 00:00:00,90008.000000,251.053200,5.000000,0.200000,36.810000,24.450000
max,51290.00000,2014-12-31 00:00:00,2015-01-07 00:00:00,99301.000000,22638.480000,14.000000,0.850000,8399.976000,933.570000
std,14806.29199,NaN,NaN,32063.693350,487.565361,2.278766,0.212280,174.340972,57.296810


- De un principio sacaremos variables que no aportan nada al modelo

In [8]:
# Modificamos la Variable objetivo

df['profit'] = df['profit'].apply(lambda x: 1 if x > 0 else 0)

In [9]:
df = df.drop(['row_id','postal_code','order_id','customer_id','customer_name','product_id','product_name'], axis = 1)
df.head()

,order_date,ship_date,ship_mode,segment,city,state,country,market,region,category,sub-category,sales,quantity,discount,profit,shipping_cost,order_priority
0,2012-07-31,2012-07-31,Same Day,Consumer,New York City,New York,United States,US,East,Technology,Accessories,2309.650,7,0.0,1,933.57,Critical
1,2013-02-05,2013-02-07,Second Class,Corporate,Wollongong,New South Wales,Australia,APAC,Oceania,Furniture,Chairs,3709.395,9,0.1,0,923.63,Critical
2,2013-10-17,2013-10-18,First Class,Consumer,Brisbane,Queensland,Australia,APAC,Oceania,Technology,Phones,5175.171,9,0.1,1,915.49,Medium
3,2013-01-28,2013-01-30,First Class,Home Office,Berlin,Berlin,Germany,EU,Central,Technology,Phones,2892.510,5,0.1,0,910.16,Medium
4,2013-11-05,2013-11-06,Same Day,Consumer,Dakar,Dakar,Senegal,Africa,Africa,Technology,Copiers,2832.960,8,0.0,1,903.04,Critical


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_date      51290 non-null  datetime64[us]
 1   ship_date       51290 non-null  datetime64[us]
 2   ship_mode       51290 non-null  str           
 3   segment         51290 non-null  str           
 4   city            51290 non-null  str           
 5   state           51290 non-null  str           
 6   country         51290 non-null  str           
 7   market          51290 non-null  str           
 8   region          51290 non-null  str           
 9   category        51290 non-null  str           
 10  sub-category    51290 non-null  str           
 11  sales           51290 non-null  float64       
 12  quantity        51290 non-null  int64         
 13  discount        51290 non-null  float64       
 14  profit          51290 non-null  int64         
 15  shipping_cost

# INICIAMOS CON EL EDA

In [11]:
# Creo una columna que diga los dias de entrega para no trabajar con fechas

df['delivery_time'] = (df['ship_date'] - df['order_date']).dt.days
df = df.drop(['order_date', 'ship_date'], axis = 1)
df

,ship_mode,segment,city,state,country,market,region,category,sub-category,sales,quantity,discount,profit,shipping_cost,order_priority,delivery_time
0,Same Day,Consumer,New York City,New York,United States,US,East,Technology,Accessories,2309.650,7,0.0,1,933.570,Critical,0
1,Second Class,Corporate,Wollongong,New South Wales,Australia,APAC,Oceania,Furniture,Chairs,3709.395,9,0.1,0,923.630,Critical,2
2,First Class,Consumer,Brisbane,Queensland,Australia,APAC,Oceania,Technology,Phones,5175.171,9,0.1,1,915.490,Medium,1
3,First Class,Home Office,Berlin,Berlin,Germany,EU,Central,Technology,Phones,2892.510,5,0.1,0,910.160,Medium,2
4,Same Day,Consumer,Dakar,Dakar,Senegal,Africa,Africa,Technology,Copiers,2832.960,8,0.0,1,903.040,Critical,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51285,Same Day,Corporate,Kure,Hiroshima,Japan,APAC,North Asia,Office Supplies,Fasteners,65.100,5,0.0,1,0.010,Medium,0
51286,Standard Class,Consumer,Houston,Texas,United States,US,Central,Office Supplies,Appliances,0.444,1,0.8,0,0.010,Medium,4
51287,Same Day,Home Office,Oxnard,California,United States,US,West,Office Supplies,Envelopes,22.920,3,0.0,1,0.010,High,0
51288,Standard Class,Home Office,Valinhos,São Paulo,Brazil,LATAM,South,Office Supplies,Binders,13.440,2,0.0,1,0.003,Medium,4


In [12]:
# Vectorizado

df['ship_mode'] = pd.factorize(df['ship_mode'])[0]
df['segment'] = pd.factorize(df['segment'])[0]
df['city'] = pd.factorize(df['city'])[0]
df['state'] = pd.factorize(df['state'])[0]
df['country'] = pd.factorize(df['country'])[0]
df['market'] = pd.factorize(df['market'])[0]
df['region'] = pd.factorize(df['region'])[0]
df['category'] = pd.factorize(df['category'])[0]
df['sub-category'] = pd.factorize(df['sub-category'])[0]
df['order_priority'] = pd.factorize(df['order_priority'])[0]

In [13]:
from sklearn.preprocessing import MinMaxScaler
x = df.drop('profit', axis =1)
scaler = MinMaxScaler()
x_scaled = pd.DataFrame(scaler.fit_transform(x), index=df.index, columns = x.columns)
x_scaled['profit'] = df['profit']
x_scaled.head()

,ship_mode,segment,city,state,country,market,region,category,sub-category,sales,quantity,discount,shipping_cost,order_priority,delivery_time,profit
0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0000,0.102006,0.461538,0.000000,1.000000,0.000000,0.000000,1
1,0.333333,0.5,0.000275,0.000915,0.006849,0.166667,0.083333,0.5,0.0625,0.163837,0.615385,0.117647,0.989353,0.000000,0.285714,0
2,0.666667,0.0,0.000550,0.001830,0.006849,0.166667,0.083333,0.0,0.1250,0.228586,0.615385,0.117647,0.980633,0.333333,0.142857,1
3,0.666667,1.0,0.000825,0.002745,0.013699,0.333333,0.166667,0.0,0.1250,0.127753,0.307692,0.117647,0.974924,0.333333,0.285714,0
4,0.000000,0.0,0.001100,0.003660,0.020548,0.500000,0.250000,0.0,0.1875,0.125122,0.538462,0.000000,0.967298,0.000000,0.142857,1


In [14]:
from sklearn.model_selection import train_test_split

x = x_scaled.drop('profit', axis =1)
y = df['profit']

x_train , x_test , y_train , y_test = train_test_split(x,y,test_size=0.2,random_state=42)

# CORREMOS EL MODELO

In [15]:
# naive bayes BERNOULLINB porque nuestra variable objetivo es binaria

from sklearn.naive_bayes import BernoulliNB

model = BernoulliNB()
model.fit(x_train,y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"binarize binarize: float or None, default=0.0Threshold for binarizing (mapping to booleans) of sample features.If None, input is presumed to already consist of binary vectors.",0.0
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [16]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(x_test)
accuracy_score(y_test,y_pred)

0.8239422889452135